# Stage 4 standardized-latent ablation

This experiment tests one hypothesis only: whether poor conditioning of the raw 384-dimensional Stage 1 CNN autoencoder latent target causes Stage 4 collapse. It does **not** change the autoencoder, decoder, normalized SPECTER2 cache, atlas-free preprocessing, immutable splits, or the `768 → 512 → ReLU → 384` projector.

Every arm uses a frozen, permanently-eval Stage 1 AE. Every decoder call receives a raw 384-dimensional Stage 1 latent. Standardized and PCA-space projector outputs are inverse-transformed before decoding. Validation alone controls early stopping and checkpoint selection; the test split is evaluated only after all selected checkpoints have been written.

## 1. Mount Drive and obtain the exact repository revision

Set `REPO_REF` to a branch, tag, or commit. Existing clones are fetched and updated; fresh sessions clone the repository. The checkout is intentionally explicit and the resolved commit is recorded later.

In [ ]:
from pathlib import Path
import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

REPO_URL = "https://github.com/neurovlm/neurovlm.git"
REPO_REF = "neurovlm_experiments"  # branch, tag, or immutable commit SHA
REPO_DIR = Path("/content/neurovlm" if IN_COLAB else Path.cwd()).resolve()

if not (REPO_DIR / ".git").is_dir():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "--all", "--tags", "--prune"], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
remote_branch = subprocess.run(
    ["git", "show-ref", "--verify", "--quiet", f"refs/remotes/origin/{REPO_REF}"],
    cwd=REPO_DIR,
).returncode == 0
if remote_branch:
    subprocess.run(["git", "merge", "--ff-only", f"origin/{REPO_REF}"], cwd=REPO_DIR, check=True)
resolved_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Repository:", REPO_DIR)
print("Configured ref:", REPO_REF)
print("Resolved commit:", resolved_commit)

## 2. Install the package and notebook dependencies

The editable install and explicit `src/` path make the checked-out source authoritative.

In [ ]:
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[metrics,viz,notebook]"
], check=True)
src_path = str(REPO_DIR / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ["PYTHONPATH"] = src_path + os.pathsep + os.environ.get("PYTHONPATH", "")
os.chdir(REPO_DIR)
print("PYTHONPATH head:", sys.path[0])

## 3. Effective experiment configuration

The optimizer, learning rate, batch size, epoch limit, lack of scheduler, gradient clipping, and early stopping match the retained Stage 4 baseline. A smoke run may cap batches, but full training does not. For a deliberate resume, either retain the active pointer on Drive or set `RESUME_EXPERIMENT_DIR` to the printed timestamped directory.

In [ ]:
BRANCHES_TO_RUN = [
    "mixed_to_pubmed", "pubmed",
    "mixed_to_nilearn", "nilearn",
    "mixed_to_neurovault", "neurovault",
]
RUN_SMOKE_TEST = True
RUN_FULL_TRAINING = True

LOSS_VARIANTS = [
    "baseline_raw",
    "standardized_mse",
    "standardized_cosine",
    "standardized_cosine_norm",
    "full_whitening",
    "pca_99_5",
]
SEED = 42
PROJECTOR_SEED = 42
EPOCHS = 100
BATCH_SIZE = 64
EVAL_BATCH_SIZE = 64
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP = 1.0
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MIN_DELTA = 0.0
SCHEDULER = "none"  # retained Stage 4 baseline has no scheduler
AMP_DTYPE = "auto"  # BF16 on compatible A100/H100; FP16 fallback
LATENT_EPSILON = 1e-4
PCA_RETAINED_VARIANCE = 0.995
COSINE_WEIGHT = 0.10
NORM_WEIGHT = 0.10
TINY_OVERFIT_N = 32
TINY_OVERFIT_STEPS = 750
TINY_OVERFIT_LR = 1e-3
SMOKE_MAX_TRAIN_BATCHES = 2
SMOKE_MAX_EVAL_BATCHES = 2
FULL_DATA_LIMIT = None
RECONSTRUCTION_EXAMPLES = 6

DRIVE_OUTPUT_BASE = Path(
    "/content/drive/MyDrive/neurovlm/stage4_standardized_latent_ablation"
    if IN_COLAB else REPO_DIR / "runs" / "stage4_standardized_latent_ablation"
)
RESUME_EXPERIMENT_DIR = None  # e.g. DRIVE_OUTPUT_BASE / "20260727T220000Z"
AUTO_RESUME_ACTIVE = True

# Optional existing Stage 4 semantic callback. It must accept
# predictions=, targets=, metadata= and return numeric metrics. When None,
# semantic checkpoint selection is omitted and recorded as unavailable.
SEMANTIC_EVALUATOR = None

assert SCHEDULER == "none"
assert set(LOSS_VARIANTS) == {
    "baseline_raw", "standardized_mse", "standardized_cosine",
    "standardized_cosine_norm", "full_whitening", "pca_99_5",
}

## 4. Environment, determinism, and branch conventions

Each variant run writes `effective_config.json`, `environment.json`, `provenance.json`, `latent_transform.pt`, `training_history.csv`, `validation_metrics.csv`, `test_metrics.csv`, `checkpoint_manifest.json`, `per_dimension_latent_diagnostics.csv`, the requested plots and reconstruction examples, and bound checkpoint files. The experiment root receives `final_comparison_table.csv`, `experiment_summary.csv`, and `experiment_summary.json`.

In [ ]:
import copy, importlib.metadata, json, platform, random, shutil, tempfile
from dataclasses import asdict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from matplotlib import pyplot as plt
from torch.utils.data import DataLoader

from neurovlm import retrieval_resources as rr
from neurovlm.atlas_free_dataset import AtlasFreeCNNDataProvider
from neurovlm.atlas_free_text import AtlasFreeContrastiveCollator, AtlasFreeTextEmbeddingLookup
from neurovlm.cnn import CNNTextToBrainModel, GenerativeTextToAELatent, autoencoder_from_payload
from neurovlm.evaluation.spatial import reconstruction_metrics
from neurovlm.evaluation.text_to_brain_audit import (
    ae_ceiling_bypass, audit_text_preprocessing, autoencoder_identity,
    frozen_ae_determinism,
)
from neurovlm.experiments.stage4_latent_ablation import (
    STAGE4_ABLATION_VARIANTS, LatentTransform, Stage4AblationTrainConfig,
    compute_stage4_ablation_loss, encode_stage1_latents,
    evaluate_stage4_ablation, resolve_amp_dtype, split_fingerprint,
    text_cache_identity, train_stage4_ablation,
)
from neurovlm.pipelines import (
    atomic_write_csv, atomic_write_json, environment_provenance,
    git_provenance, sha256_file, sha256_state_dict, sha256_value,
)
from neurovlm.training.text_to_brain import (
    _autoencoder_state_provenance, _text_cache_provenance,
    _validate_recorded_autoencoder_state, _validate_recorded_text_cache,
)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MIXED_PRECISION_DTYPE = resolve_amp_dtype(DEVICE, AMP_DTYPE)
packages = ["neurovlm", "torch", "numpy", "pandas", "matplotlib", "nilearn", "nibabel", "huggingface-hub"]
versions = {}
for package in packages:
    try: versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: versions[package] = None
ENVIRONMENT = {
    **environment_provenance(packages),
    "python_full": sys.version,
    "pytorch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_capability": torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None,
    "bf16_supported": torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    "mixed_precision_dtype": str(MIXED_PRECISION_DTYPE),
    "git": git_provenance(REPO_DIR),
    "configured_ref": REPO_REF,
    "resolved_commit": resolved_commit,
    "packages": versions,
}
print(json.dumps(ENVIRONMENT, indent=2, default=str))

The six controlled branches follow the existing NeuroVLM convention. `mixed_to_*` uses the released mixed Stage 1A AE; the domain-only name uses the released specialized Stage 1B AE.

In [ ]:
BRANCH_SPECS = {
    "mixed_to_pubmed": {"domain": "pubmed", "branch": "mixed_to_pubmed", "variant": "mixed_baseline", "stage1": "1A", "ae_variant": "mixed"},
    "pubmed": {"domain": "pubmed", "branch": "pubmed", "variant": "finetuned", "stage1": "1B", "ae_variant": "pubmed"},
    "mixed_to_nilearn": {"domain": "nilearn", "branch": "mixed_to_nilearn", "variant": "mixed_baseline", "stage1": "1A", "ae_variant": "mixed"},
    "nilearn": {"domain": "nilearn", "branch": "nilearn", "variant": "finetuned", "stage1": "1B", "ae_variant": "nilearn"},
    "mixed_to_neurovault": {"domain": "neurovault", "branch": "mixed_to_neurovault", "variant": "mixed_baseline", "stage1": "1A", "ae_variant": "mixed"},
    "neurovault": {"domain": "neurovault", "branch": "neurovault", "variant": "finetuned", "stage1": "1B", "ae_variant": "neurovault"},
}
unknown = sorted(set(BRANCHES_TO_RUN) - set(BRANCH_SPECS))
if unknown: raise ValueError(f"Unknown branches: {unknown}")
pd.DataFrame([BRANCH_SPECS[name] for name in BRANCHES_TO_RUN])

## 5. Output lifecycle and provenance helpers

The active-pointer file makes Drive resume interruption-safe. A completed pointer is never silently reused. Provenance is validated immediately with the patched Stage 4 validators and is then embedded, together with the fitted transform and effective configuration, in every checkpoint.

In [ ]:
def utc_stamp():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

def atomic_torch_save(path, payload):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    os.close(descriptor); temporary = Path(temporary_name)
    try:
        torch.save(payload, temporary)
        with temporary.open("rb") as stream: os.fsync(stream.fileno())
        os.replace(temporary, path)
    except BaseException:
        temporary.unlink(missing_ok=True)
        raise

def resolve_experiment_root():
    DRIVE_OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
    pointer = DRIVE_OUTPUT_BASE / "ACTIVE_EXPERIMENT.json"
    if RESUME_EXPERIMENT_DIR is not None:
        root = Path(RESUME_EXPERIMENT_DIR)
    elif AUTO_RESUME_ACTIVE and pointer.exists():
        active = json.loads(pointer.read_text())
        candidate = Path(active["path"])
        root = candidate if active.get("state") != "completed" and candidate.exists() else DRIVE_OUTPUT_BASE / utc_stamp()
    else:
        root = DRIVE_OUTPUT_BASE / utc_stamp()
    root.mkdir(parents=True, exist_ok=True)
    atomic_write_json(pointer, {"path": str(root), "state": "running", "updated_at": utc_stamp()})
    return root, pointer

def freeze_ae(autoencoder):
    autoencoder.eval(); autoencoder.encoder.eval(); autoencoder.decoder.eval()
    for parameter in autoencoder.parameters(): parameter.requires_grad_(False)
    return autoencoder

def load_branch_resources(branch_name):
    spec = BRANCH_SPECS[branch_name]
    filename = rr.CNN_AUTOENCODER_FILENAMES[spec["ae_variant"]]
    ae_path = Path(rr._download_from_hf(rr.ATLAS_FREE_CNN_MODEL_REPO_ID, filename, repo_type="model"))
    payload = torch.load(ae_path, map_location="cpu", weights_only=True)
    autoencoder = freeze_ae(autoencoder_from_payload(payload))
    provider = AtlasFreeCNNDataProvider(domain=spec["domain"], limit=FULL_DATA_LIMIT)
    return spec, ae_path, autoencoder, provider

def build_branch_provenance(spec, ae_path, autoencoder, provider, lookup):
    ae_source = _autoencoder_state_provenance({
        "kind": "released", "path": str(ae_path.resolve()), "sha256": sha256_file(ae_path),
        "domain": spec["domain"], "branch": spec["branch"], "stage1": spec["stage1"],
        "variant": spec["variant"], "loader_variant": spec["ae_variant"],
    }, autoencoder)
    cache_source = _text_cache_provenance(lookup)
    _validate_recorded_autoencoder_state(ae_source, autoencoder)
    _validate_recorded_text_cache(cache_source, _text_cache_provenance(lookup))
    text_audit = audit_text_preprocessing(lookup)
    if not text_audit["passed"]: raise ValueError(f"Stage 4 text-cache convention mismatch: {text_audit}")
    splits = {name: split_fingerprint(getattr(provider, name)) for name in ("train", "val", "test")}
    architecture = {
        "projector": {"name": "GenerativeTextToAELatent", "layers": [768, 512, "ReLU", 384]},
        "autoencoder": autoencoder_identity(autoencoder, checkpoint=ae_path, domain=spec["domain"], branch=spec["branch"])["architecture"],
        "decoder_input": "raw_384d_stage1_ae_latent",
    }
    return {
        "autoencoder": ae_source,
        "autoencoder_identity": autoencoder_identity(autoencoder, checkpoint=ae_path, domain=spec["domain"], branch=spec["branch"]),
        "text_cache": {**cache_source, **text_cache_identity(lookup)},
        "text_preprocessing_audit": text_audit,
        "splits": splits,
        "branch": dict(spec),
        "architecture": architecture,
        "git_commit": resolved_commit,
    }

EXPERIMENT_ROOT, ACTIVE_POINTER = resolve_experiment_root()
ROOT_CONFIG = {
    "branches_to_run": BRANCHES_TO_RUN, "loss_variants": LOSS_VARIANTS,
    "run_smoke_test": RUN_SMOKE_TEST, "run_full_training": RUN_FULL_TRAINING,
    "seed": SEED, "projector_seed": PROJECTOR_SEED, "epochs": EPOCHS,
    "batch_size": BATCH_SIZE, "eval_batch_size": EVAL_BATCH_SIZE,
    "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
    "gradient_clip": GRADIENT_CLIP, "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA, "scheduler": SCHEDULER,
    "amp_dtype": AMP_DTYPE, "latent_epsilon": LATENT_EPSILON,
    "pca_retained_variance": PCA_RETAINED_VARIANCE,
    "semantic_evaluator_enabled": callable(SEMANTIC_EVALUATOR),
}
atomic_write_json(EXPERIMENT_ROOT / "effective_config.json", ROOT_CONFIG)
atomic_write_json(EXPERIMENT_ROOT / "environment.json", ENVIRONMENT)
print("Experiment directory:", EXPERIMENT_ROOT)
print("Resume by setting RESUME_EXPERIMENT_DIR to the path above.")

## 6. Smoke, inverse, overfit, plotting, and final-test helpers

The smoke suite uses training examples only. It runs the exact AE-ceiling bypass, five-repeat frozen-AE determinism, transform/inverse audits, one 32-pair overfit for each representation, and one shuffled-pair negative control. Full-rank transforms must round-trip within tolerance; reduced PCA reports its deliberate approximation error and its own representation ceiling.

In [ ]:
def first_training_batch(dataset, lookup, n=32):
    loader = DataLoader(dataset, batch_size=n, shuffle=False, collate_fn=AtlasFreeContrastiveCollator(lookup, (36, 45, 38)))
    return next(iter(loader))

def tiny_overfit_one(variant, transform, autoencoder, batch, target_latents, shuffled=False):
    seed_everything(PROJECTOR_SEED)
    projector = GenerativeTextToAELatent(768, 512, 384).to(DEVICE)
    optimizer = torch.optim.AdamW(projector.parameters(), lr=TINY_OVERFIT_LR, weight_decay=0.0)
    text = batch["text_embedding"].to(DEVICE)
    training_text = text.roll(1, 0) if shuffled else text
    volume = batch["volume"].to(DEVICE)
    target_raw = target_latents[:len(volume)].to(DEVICE)
    transform = copy.deepcopy(transform).to(DEVICE)
    autoencoder = autoencoder.to(DEVICE).eval()
    history = []
    def snapshot(step):
        projector.eval()
        with torch.no_grad():
            represented = projector(text)
            raw = transform.inverse(represented)
            decoded = autoencoder.decoder(raw)
            spatial = reconstruction_metrics(decoded, volume)
            active = slice(0, transform.active_dim)
            transformed_target = transform.transform(target_raw)
            transformed_mse = F.mse_loss((represented * transform.active_mask)[:, active], transformed_target[:, active])
        history.append({"step": step, "transformed_latent_mse": float(transformed_mse), **spatial})
        projector.train()
    snapshot(0)
    for step in range(1, TINY_OVERFIT_STEPS + 1):
        optimizer.zero_grad(set_to_none=True)
        output = compute_stage4_ablation_loss(
            variant, projector(training_text), target_raw, volume,
            transform=transform, decoder=autoencoder.decoder,
            cosine_weight=COSINE_WEIGHT, norm_weight=NORM_WEIGHT,
        )
        output.total.backward(); optimizer.step()
        if step % 25 == 0 or step == TINY_OVERFIT_STEPS: snapshot(step)
    return {"variant": variant, "shuffled": shuffled, "initial": history[0], "final": history[-1], "history": history}

def run_smoke_suite(branch_dir, autoencoder, provider, lookup, train_latents, transforms):
    smoke_dir = branch_dir / "smoke_tests"; smoke_dir.mkdir(parents=True, exist_ok=True)
    batch = first_training_batch(provider.train, lookup, TINY_OVERFIT_N)
    volume = batch["volume"].to(DEVICE)
    model = CNNTextToBrainModel(GenerativeTextToAELatent(), autoencoder).to(DEVICE)
    ceiling = ae_ceiling_bypass(model, volume)
    determinism = frozen_ae_determinism(model, volume, repeats=5)
    if not ceiling["passed"]: raise RuntimeError(f"AE ceiling bypass failed: {ceiling}")
    if not determinism["passed"]: raise RuntimeError(f"Frozen AE determinism failed: {determinism}")
    inverse_rows = []
    with torch.no_grad():
        raw = train_latents[:len(volume)].to(DEVICE)
        raw_decoded = autoencoder.decoder(raw)
        for name, transform in transforms.items():
            fitted = copy.deepcopy(transform).to(DEVICE)
            restored = fitted.inverse(fitted.transform(raw))
            restored_decoded = autoencoder.decoder(restored)
            row = {
                "transform": name, "active_dim": fitted.active_dim,
                "retained_variance": fitted.metadata()["retained_variance"],
                "max_latent_reconstruction_error": float((restored - raw).abs().max()),
                "mean_latent_reconstruction_error": float((restored - raw).abs().mean()),
                "max_decoded_reconstruction_error": float((restored_decoded - raw_decoded).abs().max()),
                "mean_decoded_reconstruction_error": float((restored_decoded - raw_decoded).abs().mean()),
            }
            if fitted.is_lossless and row["max_latent_reconstruction_error"] > 5e-4:
                raise RuntimeError(f"Lossless transform round-trip failed: {row}")
            inverse_rows.append(row)
    representation_variants = ["baseline_raw", "standardized_mse", "full_whitening", "pca_99_5"]
    overfits = []
    for variant in representation_variants:
        overfits.append(tiny_overfit_one(variant, transforms[STAGE4_ABLATION_VARIANTS[variant]], autoencoder, batch, train_latents))
    shuffled = tiny_overfit_one("standardized_mse", transforms["standardized"], autoencoder, batch, train_latents, shuffled=True)
    atomic_write_json(smoke_dir / "ae_ceiling_bypass.json", ceiling)
    atomic_write_json(smoke_dir / "frozen_ae_determinism.json", determinism)
    atomic_write_csv(smoke_dir / "transform_inverse_audit.csv", inverse_rows)
    atomic_write_json(smoke_dir / "tiny_overfit_results.json", {"paired": overfits, "shuffled_negative": shuffled})
    atomic_write_csv(smoke_dir / "tiny_overfit_history.csv", [
        {"variant": result["variant"], "shuffled": result["shuffled"], **row}
        for result in [*overfits, shuffled] for row in result["history"]
    ])
    return {"ceiling": ceiling, "determinism": determinism, "inverse": inverse_rows, "overfits": overfits, "shuffled": shuffled}

def save_plots(run_dir, train_latents, transform, validation, examples):
    plots = run_dir / "plots"; plots.mkdir(exist_ok=True)
    x = train_latents.float(); centered = x - x.mean(0); covariance = centered.T @ centered / max(len(x) - 1, 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    image = axes[0].imshow(covariance.numpy(), cmap="coolwarm", aspect="auto"); axes[0].set_title("Training latent covariance"); fig.colorbar(image, ax=axes[0])
    axes[1].semilogy(transform.eigenvalues.clamp_min(1e-12).cpu().numpy()); axes[1].axvline(transform.active_dim - 1, color="red", ls="--"); axes[1].set_title("Covariance eigenvalues"); axes[1].set_xlabel("component")
    fig.tight_layout(); fig.savefig(plots / "covariance_eigenvalue_plots.png", dpi=160); plt.close(fig)
    diagnostics = pd.DataFrame(validation.per_dimension)
    fig, ax = plt.subplots(figsize=(11, 4)); ax.plot(diagnostics["dimension"], diagnostics["variance_ratio"]); ax.axhline(1, color="black", ls="--"); ax.set_yscale("log"); ax.set_title("Predicted / target per-dimension latent variance"); fig.tight_layout(); fig.savefig(plots / "latent_variance_ratio.png", dpi=160); plt.close(fig)
    history = pd.read_csv(run_dir / "validation_metrics.csv")
    fig, ax = plt.subplots(figsize=(8, 4)); ax.plot(history["epoch"], history["val_target_latent_norm_mean"], label="target"); ax.plot(history["epoch"], history["val_predicted_latent_norm_mean"], label="predicted"); ax.legend(); ax.set_title("Target versus predicted raw latent norm"); fig.tight_layout(); fig.savefig(plots / "target_vs_predicted_latent_norm.png", dpi=160); plt.close(fig)
    if examples:
        fig, axes = plt.subplots(len(examples), 2, figsize=(8, 3 * len(examples)), squeeze=False)
        for row, example in enumerate(examples):
            target = example["target"].squeeze(); prediction = example["prediction"].squeeze(); z = target.shape[-1] // 2
            axes[row, 0].imshow(target[:, :, z], cmap="hot"); axes[row, 0].set_title(f"target {example['map_id']}")
            axes[row, 1].imshow(prediction[:, :, z].clamp(0, 1), cmap="hot"); axes[row, 1].set_title("prediction")
            axes[row, 0].axis("off"); axes[row, 1].axis("off")
        fig.tight_layout(); fig.savefig(plots / "reconstruction_examples.png", dpi=160); plt.close(fig)

def evaluate_selected_checkpoints(run_dir, result, autoencoder, transform, provider, lookup, train_latents):
    manager = result["checkpoint_manager"]; projector = result["projector"]
    manifest = json.loads((run_dir / "checkpoint_manifest.json").read_text())
    test_rows = []; selected_validation = None; selected_examples = ()
    for role, record in manifest["checkpoints"].items():
        if role == "last": continue
        payload = manager.resume(projector, path=record["path"], map_location=DEVICE)
        evaluation = evaluate_stage4_ablation(
            projector, autoencoder, transform, provider.test, lookup,
            training_reference_latents=train_latents, device=DEVICE,
            batch_size=EVAL_BATCH_SIZE, target_shape=(36, 45, 38),
            reconstruction_examples=RECONSTRUCTION_EXAMPLES,
            semantic_evaluator=SEMANTIC_EVALUATOR,
        )
        test_rows.append({"checkpoint_role": role, "checkpoint_epoch": payload["epoch"], "n": evaluation.n, **evaluation.summary})
        if role == "top5_dice":
            selected_validation = payload["metrics"]
            selected_examples = evaluation.examples
    atomic_write_csv(run_dir / "test_metrics.csv", test_rows)
    if selected_validation is None: raise RuntimeError("No best validation top-5 Dice checkpoint was saved")
    top5_record = manifest["checkpoints"]["top5_dice"]
    manager.resume(projector, path=top5_record["path"], map_location=DEVICE)
    return selected_validation, test_rows, selected_examples

## 7. Run all requested branches and variants

Target transforms are fitted once per branch from ordered **training** AE latents only. Validation latents are never used to fit statistics. Test volumes are not evaluated until after training and all validation-selected checkpoints are fixed.

In [ ]:
lookup = AtlasFreeTextEmbeddingLookup.published()
summary_rows = []
all_branch_provenance = {}

for branch_name in BRANCHES_TO_RUN:
    print(f"\n===== {branch_name} =====")
    branch_dir = EXPERIMENT_ROOT / branch_name; branch_dir.mkdir(parents=True, exist_ok=True)
    spec, ae_path, autoencoder, provider = load_branch_resources(branch_name)
    provenance = build_branch_provenance(spec, ae_path, autoencoder, provider, lookup)
    all_branch_provenance[branch_name] = provenance
    atomic_write_json(branch_dir / "provenance.json", provenance)

    train_latents_path = branch_dir / "training_target_latents.pt"
    if train_latents_path.exists():
        cached = torch.load(train_latents_path, map_location="cpu", weights_only=True)
        if cached.get("split_sha256") != provenance["splits"]["train"]["ordered_rows_sha256"]:
            raise ValueError("Cached training latent split fingerprint mismatch")
        if cached.get("encoder_state_sha256") != provenance["autoencoder"]["encoder_state_sha256"]:
            raise ValueError("Cached training latent encoder checksum mismatch")
        train_latents = cached["latents"]
    else:
        train_latents = encode_stage1_latents(autoencoder, provider.train, lookup, device=DEVICE, batch_size=EVAL_BATCH_SIZE)
        atomic_torch_save(train_latents_path, {
            "latents": train_latents,
            "split_sha256": provenance["splits"]["train"]["ordered_rows_sha256"],
            "encoder_state_sha256": provenance["autoencoder"]["encoder_state_sha256"],
        })
    if len(train_latents) != len(provider.train): raise RuntimeError("Training latent alignment failure")

    transforms = {
        "raw": LatentTransform.fit(train_latents, "raw", epsilon=LATENT_EPSILON),
        "standardized": LatentTransform.fit(train_latents, "standardized", epsilon=LATENT_EPSILON),
        "full_whitening": LatentTransform.fit(train_latents, "full_whitening", epsilon=LATENT_EPSILON),
        "pca_99_5": LatentTransform.fit(train_latents, "pca_99_5", epsilon=LATENT_EPSILON, retained_variance=PCA_RETAINED_VARIANCE),
    }
    atomic_write_json(branch_dir / "latent_transform_summary.json", {name: transform.metadata() for name, transform in transforms.items()})

    if RUN_SMOKE_TEST:
        smoke = run_smoke_suite(branch_dir, autoencoder, provider, lookup, train_latents, transforms)
        print("Smoke tests passed; PCA active dimensions:", transforms["pca_99_5"].active_dim)

    if RUN_FULL_TRAINING:
        for variant in LOSS_VARIANTS:
            print(f"--- {branch_name} / {variant} ---")
            transform = transforms[STAGE4_ABLATION_VARIANTS[variant]]
            run_dir = branch_dir / variant; run_dir.mkdir(parents=True, exist_ok=True)
            config = Stage4AblationTrainConfig(
                variant=variant, seed=SEED, projector_seed=PROJECTOR_SEED,
                epochs=EPOCHS, batch_size=BATCH_SIZE, eval_batch_size=EVAL_BATCH_SIZE,
                learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
                gradient_clip=GRADIENT_CLIP,
                early_stopping_patience=EARLY_STOPPING_PATIENCE,
                early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
                amp=True, amp_dtype=AMP_DTYPE,
                cosine_weight=COSINE_WEIGHT, norm_weight=NORM_WEIGHT,
                scheduler=SCHEDULER, reconstruction_examples=RECONSTRUCTION_EXAMPLES,
            )
            effective = {**config.effective_dict(), "branch": spec, "transform": transform.metadata()}
            run_provenance = {**provenance, "latent_transform": transform.metadata()}
            binding = {
                "autoencoder": provenance["autoencoder"],
                "text_cache": provenance["text_cache"],
                "splits": provenance["splits"],
                "branch": provenance["branch"],
                "architecture": provenance["architecture"],
                "latent_transform": transform.metadata(),
            }
            atomic_write_json(run_dir / "effective_config.json", effective)
            atomic_write_json(run_dir / "environment.json", ENVIRONMENT)
            atomic_write_json(run_dir / "provenance.json", run_provenance)
            atomic_torch_save(run_dir / "latent_transform.pt", transform.to_payload())
            result = train_stage4_ablation(
                config, run_dir=run_dir, autoencoder=autoencoder,
                transform=transform, training_latents=train_latents,
                train_dataset=provider.train, validation_dataset=provider.val,
                lookup=lookup, binding=binding, device=DEVICE,
                semantic_evaluator=SEMANTIC_EVALUATOR,
            )
            selected_val, test_rows, selected_examples = evaluate_selected_checkpoints(
                run_dir, result, autoencoder, transform, provider, lookup, train_latents
            )
            selected_test = next(row for row in test_rows if row["checkpoint_role"] == "top5_dice")
            selected_eval = evaluate_stage4_ablation(
                result["projector"], autoencoder, transform, provider.val, lookup,
                training_reference_latents=train_latents, device=DEVICE,
                batch_size=EVAL_BATCH_SIZE, reconstruction_examples=RECONSTRUCTION_EXAMPLES,
                semantic_evaluator=SEMANTIC_EVALUATOR,
            )
            atomic_write_csv(run_dir / "per_dimension_latent_diagnostics.csv", selected_eval.per_dimension)
            save_plots(run_dir, train_latents, transform, selected_eval, selected_examples)
            summary_rows.append({
                "branch": branch_name, "domain": spec["domain"], "stage1": spec["stage1"],
                "variant": variant, "transform": transform.kind,
                "active_dim": transform.active_dim,
                "retained_pca_variance": transform.metadata()["retained_variance"],
                "epochs_completed": result["epochs_completed"],
                **{key: value for key, value in selected_val.items() if str(key).startswith("val_")},
                **{f"test_{key}": value for key, value in selected_test.items() if key not in {"checkpoint_role"}},
                "run_dir": str(run_dir),
            })
            atomic_write_csv(EXPERIMENT_ROOT / "experiment_summary.csv", summary_rows)
            atomic_write_json(EXPERIMENT_ROOT / "experiment_summary.json", summary_rows)
    del autoencoder, provider, train_latents, transforms
    if torch.cuda.is_available(): torch.cuda.empty_cache()

atomic_write_json(EXPERIMENT_ROOT / "provenance.json", all_branch_provenance)
print("Completed requested work in", EXPERIMENT_ROOT)

## 8. Final comparison and requested conclusions

The unchanged `baseline_raw` arm is the within-branch reference. Deltas below use the best validation top-5 Dice checkpoint for each arm. This preserves the retained selection rule and prevents post-hoc test selection.

In [ ]:
summary = pd.DataFrame(summary_rows if summary_rows else pd.read_csv(EXPERIMENT_ROOT / "experiment_summary.csv").to_dict("records"))
if summary.empty:
    print("No full-training rows exist. Set RUN_FULL_TRAINING=True to produce conclusions.")
else:
    key_metrics = [
        "val_predicted_target_latent_variance_ratio", "val_global_explained_variance",
        "val_spatial_corr", "val_top5_dice", "val_foreground_mse",
    ]
    baselines = summary[summary.variant == "baseline_raw"].set_index("branch")
    for metric in key_metrics:
        if metric in summary and metric in baselines:
            summary[f"delta_{metric}_vs_baseline"] = summary.apply(
                lambda row: row[metric] - baselines.loc[row.branch, metric], axis=1
            )
    comparison_path = EXPERIMENT_ROOT / "final_comparison_table.csv"
    summary.to_csv(comparison_path, index=False)
    atomic_write_csv(EXPERIMENT_ROOT / "experiment_summary.csv", summary.to_dict("records"))
    atomic_write_json(EXPERIMENT_ROOT / "experiment_summary.json", summary.to_dict("records"))
    display(summary.sort_values(["branch", "val_top5_dice"], ascending=[True, False]))

    best = summary.loc[summary.groupby("branch")["val_top5_dice"].idxmax()].copy()
    conditioned = summary[summary.variant != "baseline_raw"]
    sparse = conditioned[conditioned.domain.isin(["pubmed", "nilearn"])]
    neurovault = conditioned[conditioned.domain == "neurovault"]
    pca = summary[summary.variant == "pca_99_5"]
    answers = {
        "1_latent_variance_retention": {
            "mean_delta_vs_raw": float(conditioned["delta_val_predicted_target_latent_variance_ratio_vs_baseline"].mean()),
            "improved_branch_variant_count": int((conditioned["delta_val_predicted_target_latent_variance_ratio_vs_baseline"] > 0).sum()),
            "interpretation": "Positive values indicate that conditioning increased raw Stage 1 latent variance retention.",
        },
        "2_spatial_generation": {
            "mean_top5_dice_delta": float(conditioned["delta_val_top5_dice_vs_baseline"].mean()),
            "mean_spatial_corr_delta": float(conditioned["delta_val_spatial_corr_vs_baseline"].mean()),
            "interpretation": "Spatial improvement requires positive Dice/correlation deltas, not latent metrics alone.",
        },
        "3_sparse_vs_neurovault": {
            "pubmed_nilearn_mean_top5_delta": float(sparse["delta_val_top5_dice_vs_baseline"].mean()),
            "neurovault_mean_top5_delta": float(neurovault["delta_val_top5_dice_vs_baseline"].mean()),
            "sparse_improves_more": bool(sparse["delta_val_top5_dice_vs_baseline"].mean() > neurovault["delta_val_top5_dice_vs_baseline"].mean()),
        },
        "4_pca_spatial_information": {
            "mean_pca_top5_delta": float(pca["delta_val_top5_dice_vs_baseline"].mean()),
            "mean_pca_spatial_corr_delta": float(pca["delta_val_spatial_corr_vs_baseline"].mean()),
            "active_dimensions_by_branch": dict(zip(pca.branch, pca.active_dim)),
            "interpretation": "Negative PCA spatial deltas despite retained variance indicate discarded spatially important directions.",
        },
        "5_best_validation_variant": best[["branch", "variant", "val_top5_dice", "delta_val_top5_dice_vs_baseline", "val_spatial_corr", "val_global_explained_variance"]].to_dict("records"),
    }
    atomic_write_json(EXPERIMENT_ROOT / "final_answers.json", answers)
    print(json.dumps(answers, indent=2))
    atomic_write_json(ACTIVE_POINTER, {"path": str(EXPERIMENT_ROOT), "state": "completed", "updated_at": utc_stamp()})

### Interpretation guardrails

1. A conditioning explanation is supported only if standardized/whitened arms materially increase **raw** latent variance retention or explained variance on validation data.
2. Better transformed-space loss alone is insufficient; spatial correlation and top-1/5/10 Dice must also improve.
3. The PubMed/Nilearn-versus-NeuroVault comparison must be made from within-branch deltas because the domains and Stage 1 ceilings differ.
4. The reduced PCA arm is intentionally not lossless. Its transform audit quantifies latent and decoded approximation error before training, allowing spatial degradation to be attributed to discarded directions rather than a broken inverse.
5. Test metrics describe the already-selected validation checkpoints. They never choose a variant, epoch, or checkpoint.